# 05 - Reranker experiment (retrieval only)

Does a better reranker put the right chunk in the top 5 or top 8? No generation here, so it is cheap.
All rerankers order the **same** candidate pool of `rag8` (company/year filter, dense + BM25).
Existing systems and results are untouched and stay as the benchmark.

Needs a Cohere trial key stored as a Colab Secret named `COHERE_API_KEY` (never paste it in a cell).
Trial limits (check them on the Cohere site): about 10 rerank calls a minute and 1,000 calls a month.
This notebook makes one call per question and per model (155 each), and caches every answer.

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

In [ ]:
import numpy as np

from src.eval.golden import load_golden, validate
from src.hub.dataset import download_folder
from src.retrieval.embed import embed_texts, load_chunks

HF_REPO = "IlariaCesa/10k-agentic-rag-data"
download_folder(HF_REPO, "text", Path("data"))
chunks = load_chunks("data/text/chunks_1600.jsonl")
vectors = np.load("data/text/embeddings_bge-m3.npy").astype(np.float32)
assert len(chunks) == len(vectors)
all_questions = load_golden(Path("questions/questions.jsonl"))
errors, _ = validate(all_questions, chunks)
assert not errors, errors
answerable = [q for q in all_questions if q["expected"] == "answer"]
print(len(answerable), "answerable questions")

In [ ]:
%pip install -q cohere
from sentence_transformers import SentenceTransformer

from src.retrieval.pipeline import Corpus
from src.retrieval.rerank import Reranker
from src.retrieval.rerank_api import CohereReranker

model = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
model.max_seq_length = 1024
query_vectors = embed_texts(model, [q["question"] for q in answerable], batch_size=32)
corpus = Corpus(chunks, vectors)

os.makedirs("results", exist_ok=True)
COHERE_MODELS = ["rerank-v3.5", "rerank-v4.0-pro"]  # names from Cohere's changelog; edit if needed
key = userdata.get("COHERE_API_KEY")  # the key stays in the Colab Secret
scorers = {"bge-reranker-v2-m3": Reranker(max_length=1024)}
for name in COHERE_MODELS:
    scorers[name] = CohereReranker(
        api_key=key,
        model=name,
        cache_path=f"results/cohere_cache_{name}.jsonl",
        max_calls=len(answerable) + 5,
    )

In [ ]:
from src.eval.rerank_experiment import evaluate

rows = evaluate(corpus, answerable, query_vectors, scorers)
df = pd.DataFrame(rows)
for name in COHERE_MODELS:
    print(name, "- real API calls in this run:", scorers[name].calls)
df.to_json("results/rerank_experiment.jsonl", orient="records", lines=True)

In [ ]:
cols = ["hit@1", "hit@5", "hit@8", "recall@5", "recall@8", "all@5", "all@8", "mrr"]
pool = df[df["reranker"] == df["reranker"].iloc[0]]
print(
    "Pool ceiling: all gold chunks inside the pool for",
    round(pool["pool_has_all"].mean(), 3),
    "of questions; mean pool size",
    round(pool["pool_size"].mean(), 1),
)
print(df.groupby("reranker")[cols].mean().round(3).to_string())
for split in ["dev", "test", "agentic"]:
    print("\n" + split, "(", int((pool["split"] == split).sum()), "questions )")
    print(df[df["split"] == split].groupby("reranker")[cols].mean().round(3).to_string())
print("\nall@5 by question type")
print(df.groupby(["type", "reranker"])["all@5"].mean().unstack().round(3).to_string())

In [ ]:
from src.eval.grades_report import sign_test

base = "bge-reranker-v2-m3"
for metric in ["all@5", "all@8", "hit@5"]:
    wide = df.pivot(index="id", columns="reranker", values=metric)
    for name in COHERE_MODELS:
        better = int(((wide[name] == 1) & (wide[base] == 0)).sum())
        worse = int(((wide[name] == 0) & (wide[base] == 1)).sum())
        print(
            f"{metric}: {name} better on {better}, {base} better on {worse}, "
            f"sign test p={sign_test(better, worse):.3f}"
        )

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("rerank_experiment", "zip", "results")  # includes the caches: keep them
files.download("rerank_experiment.zip")